# 💬 Minería de Opinión y Análisis de Sentimiento con Léxicos en Español

### Autor: Néstor León | Business Intelligence & Analytics
---

El **análisis de sentimiento** convierte la voz del cliente en métricas cuantitativas que la dirección puede monitorizar en tiempo real. En este proyecto adaptamos la metodología utilizada con el léxico **NRC** y diccionarios de polaridad en castellano para extraer:
1. **Polaridad general**: Sentimiento positivo, neutro o negativo.
2. **Emociones primarias**: Detección de enfado (*anger*), confianza (*trust*), tristeza o alegría en el texto.
3. **Indicador de Satisfacción del Cliente (NPS / CSAT Predictivo)**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

print("Módulos inicializados.")


## 1. Construcción del Diccionario de Sentimiento y Emociones en Castellano

Definimos un léxico de referencia representativo con ponderaciones de polaridad $(-1 \dots +1)$ y categorías emocionales.


In [ ]:
lexico_sentimientos = {
    # Negativas / Foco de insatisfacción
    'pesimo': {'polaridad': -1.0, 'emocion': 'ira'},
    'malo': {'polaridad': -0.7, 'emocion': 'tristeza'},
    'mala': {'polaridad': -0.7, 'emocion': 'tristeza'},
    'indebido': {'polaridad': -0.8, 'emocion': 'ira'},
    'esperando': {'polaridad': -0.4, 'emocion': 'frustracion'},
    'fallo': {'polaridad': -0.7, 'emocion': 'frustracion'},
    'falla': {'polaridad': -0.7, 'emocion': 'frustracion'},
    'averia': {'polaridad': -0.6, 'emocion': 'preocupacion'},
    'caida': {'polaridad': -0.6, 'emocion': 'preocupacion'},
    'duplicada': {'polaridad': -0.5, 'emocion': 'ira'},
    
    # Positivas / Foco de satisfacción
    'excelente': {'polaridad': 1.0, 'emocion': 'alegria'},
    'genial': {'polaridad': 0.9, 'emocion': 'alegria'},
    'satisfecho': {'polaridad': 0.8, 'emocion': 'confianza'},
    'rapida': {'polaridad': 0.6, 'emocion': 'confianza'},
    'rapido': {'polaridad': 0.6, 'emocion': 'confianza'},
    'profesional': {'polaridad': 0.8, 'emocion': 'confianza'},
    'resolvio': {'polaridad': 0.7, 'emocion': 'confianza'},
    'buena': {'polaridad': 0.6, 'emocion': 'confianza'},
    'competitivas': {'polaridad': 0.5, 'emocion': 'confianza'}
}

print(f"Palabras en el léxico especializado: {len(lexico_sentimientos)}")


## 2. Algoritmo de Evaluación de Sentimiento

Para cada comentario:
1. Normalizamos el texto.
2. Buscamos coincidencias con el diccionario léxico.
3. Calculamos la polaridad neta del documento y registramos el desglose de emociones.


In [ ]:
def analizar_sentimiento_texto(texto):
    palabras = re.findall(r'\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]+\b', texto.lower())
    scores = []
    emociones_detectadas = []
    
    for w in palabras:
        if w in lexico_sentimientos:
            scores.append(lexico_sentimientos[w]['polaridad'])
            emociones_detectadas.append(lexico_sentimientos[w]['emocion'])
            
    if len(scores) == 0:
        return {'score': 0.0, 'clasificacion': 'Neutro', 'emocion_principal': 'Ninguna'}
    
    score_medio = np.mean(scores)
    if score_medio > 0.2:
        clasif = 'Positivo'
    elif score_medio < -0.2:
        clasif = 'Negativo'
    else:
        clasif = 'Neutro'
        
    emocion_top = pd.Series(emociones_detectadas).mode()[0] if len(emociones_detectadas) > 0 else 'Ninguna'
    
    return {
        'score': round(score_medio, 3),
        'clasificacion': clasif,
        'emocion_principal': emocion_top
    }

# Evaluamos un lote de reseñas
reviews = [
    "Excelente instalacion, el tecnico resolvio todo muy rapido y quede totalmente satisfecho.",
    "El servicio es pesimo, cobro indebido y estuve esperando dos horas con una averia en la linea.",
    "La aplicacion movil es genial y me parece muy profesional la gestion comercial.",
    "Mala conexion, caida constante de fibra y atencion al cliente muy mala.",
    "Contrate una tarifa estandar y por ahora todo correcto y normal."
]

resultados = [analizar_sentimiento_texto(r) for r in reviews]
df_analisis = pd.DataFrame(resultados)
df_analisis['comentario'] = reviews
df_analisis[['comentario', 'score', 'clasificacion', 'emocion_principal']]


## 3. Visualización Ejecutiva de Sentimiento

Generamos un gráfico de barras para monitorizar la distribución del sentimiento en el canal de atención al cliente.


In [ ]:
plt.figure(figsize=(8, 4))
palette_colors = {'Positivo': '#2ecc71', 'Neutro': '#95a5a6', 'Negativo': '#e74c3c'}
sns.countplot(data=df_analisis, x='clasificacion', palette=palette_colors)
plt.title("Distribución de Sentimiento de Clientes", fontsize=13, fontweight='bold')
plt.xlabel("Categoría de Sentimiento")
plt.ylabel("Número de Comentarios")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


## 4. Conclusiones y Próximos Pasos

- **Alineación con KPIs de Negocio**: Este pipeline de scoring permite crear alertas tempranas: si el ratio de sentimiento negativo supera el 20% en un día, se dispara un aviso para revisar caídas de servicio o problemas de facturación.
- **Evolución**: En un entorno productivo más complejo, este enfoque léxico puede combinarse con modelos basados en embeddings o *Transformers* (`BETO` / RoBERTa en español) para detectar ironías y dobles sentidos.
